# 04. Task B: hỏi đáp Software Supply Chain trên một tổ chức tổng hợp

**Câu hỏi nghiên cứu.** Cùng một reader, cùng một bộ câu hỏi, cùng ngân sách bằng chứng: khi nào LLM thuần đủ, khi nào cần truy hồi văn bản,
khi nào cần đồ thị, và khi nào hệ thống phải biết từ chối?

| Nhóm | Câu hỏi | Bằng chứng cần |
|---|---|---|
| G1 | hiểu mã (luồng dữ liệu, host, biến môi trường) | đã có trong đoạn mã đi kèm |
| G2 | chính sách nội bộ, ngoại lệ (ADR), bản cũ và bản hiện hành | tài liệu ACME |
| G3 | tác động dependency nhiều bước (service -> package -> ... -> package độc) | nhiều tài liệu nối nhau |
| G4 | thiếu bằng chứng (thực thể không tồn tại, version không pin, plugin theo config, hai policy mâu thuẫn, dynamic dispatch) | đáp án đúng là `insufficient_evidence` |

**Không thiết kế để mỗi phương pháp thắng một nhóm.** Mọi method trả lời **mọi** câu hỏi; mỗi nhóm có nhiều loại câu hỏi và nhiều bẫy
(policy cũ, dev-only, extra không bật, version ngoài khoảng, service đã ngừng, tên gần giống). Đáp án và bằng chứng chuẩn tính từ tổ chức sinh ra,
không từ model. KG được **parse từ chính các tài liệu** mà text RAG dùng, nên hai bên có cùng thông tin.
Tổ chức seed 1 (dev) chỉ để tune chính sách E4; mọi con số báo cáo lấy từ seed 2 (test).

In [1]:
# ==== 00a. Mount Drive + đường dẫn + helper ====
import os, sys, json, subprocess, time, shutil, glob, random
try:
    from google.colab import drive; drive.mount('/content/drive'); IN_COLAB = True
except ImportError:
    IN_COLAB = False                                        # chạy local để kiểm tra notebook (CPU)

DRIVE_BASE = '/content/drive/MyDrive/ThS/Research/colab' if IN_COLAB else os.environ.get('PYPI_RAG_BASE', os.path.abspath('../..'))  # SỬA NẾU KHÁC
CODE_DIR = f'{DRIVE_BASE}/pypi_malware_rag'                 # folder code này (upload nguyên folder lên Drive)
WORK     = os.environ.get('PYPI_RAG_WORK', f'{DRIVE_BASE}/pypi_rag_work')   # dữ liệu chuẩn hoá, KB, kết quả (giữ qua các phiên)
EXT      = '/content/external' if IN_COLAB else f'{WORK}/external'          # repo upstream + package nghi độc: đĩa máy ảo, KHÔNG để trên Drive
QUICK    = os.environ.get('PYPI_RAG_QUICK') == '1'          # 1 = kiểm tra nhanh: chỉ dữ liệu tổng hợp, reader dummy, không tải gì

os.environ['PYTHONDONTWRITEBYTECODE'] = '1'; os.environ['PYTHONIOENCODING'] = 'utf-8'; sys.dont_write_bytecode = True
for d in glob.glob(f'{CODE_DIR}/**/__pycache__', recursive=True): shutil.rmtree(d, ignore_errors=True)
for d in (WORK, f'{WORK}/data', f'{WORK}/kb', f'{WORK}/results', EXT): os.makedirs(d, exist_ok=True)
os.chdir(CODE_DIR); sys.path.insert(0, CODE_DIR)
assert os.path.exists('pypi_rag/cli.py'), 'chưa upload folder pypi_malware_rag lên Drive (hoặc sai DRIVE_BASE)'
for m in [m for m in sys.modules if m.split('.')[0] == 'pypi_rag']: del sys.modules[m]

CLI = f'{sys.executable} -m pypi_rag.cli'
def sh(cmd, check=True):
    """Chạy lệnh, in output trực tiếp; lỗi -> exception (check=True)."""
    print('$', cmd[:500]); t0 = time.time()
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         encoding='utf-8', errors='replace', cwd=CODE_DIR)
    for line in p.stdout: print(line, end='')
    rc = p.wait(); print(f'exit={rc}  ({(time.time() - t0) / 60:.1f} phút)')
    if check and rc: raise RuntimeError(f'lệnh lỗi (exit {rc}): {cmd[:200]}')
    return rc
def gate(cond, msg): print(('PASS  ' if cond else 'FAIL  ') + msg)
def load(out): return json.load(open(f'{out}/results.json', encoding='utf-8'))
import numpy as np, pandas as pd
pd.set_option('display.max_colwidth', 120); pd.set_option('display.width', 200)
print('code =', CODE_DIR, '| work =', WORK, '| colab =', IN_COLAB, '| quick =', QUICK)

code = /workspace/code/pypi_malware_rag | work = /workspace/pypi_rag_work | colab = False | quick = False


In [2]:
# ==== 00b. GPU + HF token + backend ====
BACKEND = os.environ.get('PYPI_RAG_BACKEND', 'vllm')   # 'vllm' (nhanh nhất) | 'hf' (transformers, chậm hơn 4-8 lần, không cần cài thêm)
if IN_COLAB:
    sh(f'{sys.executable} -m pip install -q "sentence-transformers>=2.2" "transformers>=4.43" accelerate networkx pyyaml')
if BACKEND == 'vllm' and not QUICK:
    ok = lambda: subprocess.call([sys.executable, '-c', 'import vllm'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL) == 0
    if not ok():
        sh(f'{sys.executable} -m pip install -q vllm', check=False)   # lần đầu 3-5 phút; nếu sau đó kernel báo lỗi thư viện: Restart session và chạy lại
    if not ok():
        print('[vllm] không cài được -> dùng backend hf'); BACKEND = 'hf'
print('backend =', BACKEND)
if IN_COLAB:
    try:
        from google.colab import userdata
        if userdata.get('HF_TOKEN'): os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN'); print('HF_TOKEN set')
    except Exception: print('(không có secret HF_TOKEN: dùng mirror NousResearch cho Llama)')
try:
    import torch
    print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
    if not QUICK: assert torch.cuda.is_available(), 'cần GPU: Runtime > Change runtime type > A100'
except ImportError:
    print('(không có torch: chỉ chạy được QUICK mode)')

backend = vllm


torch 2.14.0+cu130 | GPU: NVIDIA L40S


In [3]:
# ==== B0. Cấu hình ====
READERS = ['NousResearch/Meta-Llama-3.1-8B-Instruct', 'Qwen/Qwen2.5-Coder-7B-Instruct']
if os.environ.get('PYPI_RAG_READERS'): READERS = os.environ['PYPI_RAG_READERS'].split(',')
EMB = 'BAAI/bge-small-en-v1.5'
N_TEST, N_DEV = 100, 40                         # câu hỏi mỗi nhóm
SC_METHODS = ['e0', 'e1', 'e2', 'e2_bm25', 'e3', 'e3a_exact', 'e3a', 'e4']
if QUICK: READERS, EMB, N_TEST, N_DEV = ['dummy'], 'hash', 30, 15
SC = f'{WORK}/supplychain'
def tag(rd): return rd.split('/')[-1]
def llm_of(rd): return 'dummy' if rd == 'dummy' else f'{BACKEND}:{rd}'
NAMES = {'e0': 'E0 LLM thuần', 'e1': 'E1 RAG dense', 'e2': 'E2 RAG hybrid', 'e2_bm25': 'E2 BM25', 'e3': 'E3 KG-RAG',
         'e3a_exact': 'E3a cùng cạnh, câu rời', 'e3a': 'E3b fact chọn bằng truy hồi', 'e4': 'E4 KG + kiểm chứng'}
print('readers =', READERS, '| N_TEST =', N_TEST, '| methods =', SC_METHODS)

readers = ['NousResearch/Meta-Llama-3.1-8B-Instruct', 'Qwen/Qwen2.5-Coder-7B-Instruct'] | N_TEST = 100 | methods = ['e0', 'e1', 'e2', 'e2_bm25', 'e3', 'e3a_exact', 'e3a', 'e4']


## B1. Sinh tổ chức, tài liệu, KG và câu hỏi
Mỗi seed cho một công ty khác (service, package, version, advisory, policy, ADR khác nhau). Bảng dưới: số câu hỏi theo loại và theo bẫy,
thống kê KG (mọi cạnh có `doc:line`), và các advisory cùng tập service bị ảnh hưởng (ground truth).

In [4]:
for name, seed, n in (('dev', 1, N_DEV), ('test', 2, N_TEST)):
    out = f'{SC}/{name}'
    if not os.path.exists(f'{out}/stats.json'):
        sh(f'{CLI} sc-generate --seed {seed} --out {out} --n-per-group {n}')
ST = {n: json.load(open(f'{SC}/{n}/stats.json')) for n in ('dev', 'test')}
display(pd.DataFrame({n: s['by_type'] for n, s in ST.items()}).fillna(0).astype(int))
display(pd.DataFrame({n: s['by_trap'] for n, s in ST.items()}).fillna(0).astype(int))
print('KG test:', json.dumps({k: ST['test']['kg'][k] for k in ('nodes', 'edges', 'edges_missing_provenance')}))
display(pd.DataFrame(ST['test']['advisories']).T)
gate(ST['test']['kg']['edges_missing_provenance'] == 0, 'mọi cạnh KG có provenance doc:line')

$ /usr/bin/python3 -m pypi_rag.cli sc-generate --seed 1 --out /workspace/pypi_rag_work/supplychain/dev --n-per-group 40


{
 "n_docs": 342,
 "n_questions": 160,
 "by_group": {
  "G1": 40,
  "G2": 40,
  "G3": 40,
  "G4": 40
 },
 "kg": {
  "nodes": 423,
  "edges": 919,
  "node_kinds": {
   "pkg": 96,
   "pv": 270,
   "svc": 32,
   "team": 6,
   "adv": 7,
   "pol": 4,
   "adr": 4,
   "inc": 4
  },
  "edge_types": {
   "VERSION_OF": 270,
   "REQUIRES": 294,
   "OWNED_BY": 32,
   "CALLS": 39,
   "DEPENDS_ON": 247,
   "AFFECTS": 7,
   "MATCHES": 7,
   "BANS": 4,
   "SUPERSEDES": 1,
   "EXEMPTS": 8,
   "INVOLVES": 4,
   "MENTIONS": 6
  },
  "edges_missing_provenance": 0
 }
}
exit=0  (0.1 phút)
$ /usr/bin/python3 -m pypi_rag.cli sc-generate --seed 2 --out /workspace/pypi_rag_work/supplychain/test --n-per-group 100


{
 "n_docs": 342,
 "n_questions": 400,
 "by_group": {
  "G1": 100,
  "G2": 100,
  "G3": 100,
  "G4": 100
 },
 "kg": {
  "nodes": 423,
  "edges": 949,
  "node_kinds": {
   "pkg": 96,
   "pv": 270,
   "svc": 32,
   "team": 6,
   "adv": 7,
   "pol": 4,
   "adr": 4,
   "inc": 4
  },
  "edge_types": {
   "VERSION_OF": 270,
   "REQUIRES": 315,
   "OWNED_BY": 32,
   "DEPENDS_ON": 245,
   "CALLS": 50,
   "AFFECTS": 7,
   "MATCHES": 7,
   "BANS": 4,
   "SUPERSEDES": 1,
   "EXEMPTS": 8,
   "INVOLVES": 4,
   "MENTIONS": 6
  },
  "edges_missing_provenance": 0
 }
}
exit=0  (0.1 phút)


,dev,test
G1/code_flow,29,77
G1/code_env,4,5
G1/code_host,7,18
G2/ban_allowed,25,62
G2/exception_valid,3,4
G2/patch_window,8,23
G2/token_lifetime,4,11
G3/affected_set,3,7
G3/affected_filtered,3,7
G3/teams_to_notify,3,7


,dev,test
none,34,72
co_occurrence,6,17
cross_function,13,39
superseded_policy,16,47
dev_exempt,1,3
adr_exception,2,3
adr_expired,2,2
exception_other_service,1,4
version_out_of_range,6,18
partial_overlap,4,11


KG test: {"nodes": 423, "edges": 949, "edges_missing_provenance": 0}


,package,spec,kind,affected
ADV-2026-042,grpciio,*,typosquat,"[svc-admin, svc-gateway, svc-loyalty, svc-signup, svc-warehouse]"
ADV-2026-094,six-py,*,typosquat,"[svc-payouts, svc-wallet]"
ADV-2026-095,pygmentts,*,typosquat,"[svc-invoices, svc-partner-api]"
ADV-2026-060,botcoore,*,typosquat,"[svc-chatbot, svc-search-api]"
ADV-2026-025,urllib3,==4.9.1,compromised,"[svc-orders-api, svc-payouts]"
ADV-2026-066,tenacity,">=4.14.1,<4.15.0",compromised,"[svc-loyalty, svc-payouts]"
ADV-2026-087,acme-cache,==2.2.1,compromised,"[svc-email-relay, svc-invoices, svc-mobile-bff]"


PASS  mọi cạnh KG có provenance doc:line


## B2. Chạy: dev (tune E4) rồi test
Mỗi reader: chạy 8 method trên dev, tune `tau` của E4 trên dev (độ chính xác tổng G1-G4), rồi chạy test với chính sách đã cố định.
E4 dùng lại đúng câu trả lời của E3 (cùng prompt), chỉ thêm bước kiểm chứng; không tốn thêm GPU.

In [5]:
def sc_run(out, data, llm, extra=''):
    if os.path.exists(f'{out}/results.json'):
        print('[skip]', out); return load(out)
    sh(f'{CLI} sc-run --data {data} --out {out} --llm {llm} --embedder {EMB} --methods {" ".join(SC_METHODS)} {extra}')
    return load(out)
RB = {}
for rd in READERS:
    dev_out = f'{SC}/runs/dev_{tag(rd)}'
    sc_run(dev_out, f'{SC}/dev', llm_of(rd))
    pol = f'{dev_out}/policy.json'
    if not os.path.exists(pol):
        sh(f'{CLI} sc-tune --results {dev_out} --data {SC}/dev --out {pol}')
    print('chính sách E4 (tune trên dev):', json.load(open(pol)))
    RB[rd] = sc_run(f'{SC}/runs/test_{tag(rd)}', f'{SC}/test', llm_of(rd), f'--policy {pol}')

$ /usr/bin/python3 -m pypi_rag.cli sc-run --data /workspace/pypi_rag_work/supplychain/dev --out /workspace/pypi_rag_work/supplychain/runs/dev_Meta-Llama-3.1-8B-Instruct --llm vllm:NousResearch/Meta-Llama-3.1-8B-Instruct --embedder BAAI/bge-small-en-v1.5 --methods e0 e1 e2 e2_bm25 e3 e3a_exact e3a e4 



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4567.91it/s]


[sc] 160 questions {'G1': 40, 'G2': 40, 'G3': 40, 'G4': 40} | methods ['e0', 'e1', 'e2', 'e2_bm25', 'e3', 'e3a_exact', 'e3a', 'e4'] | llm vllm:NousResearch/Meta-Llama-3.1-8B-Instruct | 342 docs | KG 423 nodes 919 edges | policy {'tau': 0.5, 'filter_elements': True, 'untuned': True}


[sc] built 1120 prompts (8 methods x 160 questions) in 5s


W1109 19:09:03.111000 2364 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1109 19:09:03.111000 2364 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1109 19:09:03.112000 2364 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead


(MainProcess pid=2364) INFO 11-09 19:09:05 [startup.py:96] non-default args: {'dtype': 'bfloat16', 'seed': 42, 'max_model_len': 8704, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'NousResearch/Meta-Llama-3.1-8B-Instruct'}


... (281 outputs left out)

[e3a_exact] acc all 0.583 G1 0.720 G2 0.580 G3 0.540 G4 0.490 | abst G4 0.490 | false-abst 0.093 | fabricated 0.013 | unsupported 0.125
[e3a] acc all 0.465 G1 0.790 G2 0.360 G3 0.150 G4 0.560 | abst G4 0.560 | false-abst 0.320 | fabricated 0.005 | unsupported 0.048


[e4] acc all 0.615 G1 0.730 G2 0.560 G3 0.520 G4 0.650 | abst G4 0.650 | false-abst 0.127 | fabricated 0.013 | unsupported 0.030


[done] /workspace/pypi_rag_work/supplychain/runs/test_Qwen2.5-Coder-7B-Instruct/results.json (8.4 min)
(MainProcess pid=5084) INFO 11-09 19:38:51 [utils.py:628] [shutdown] Process manager: send sigterm to process EngineCore
(EngineCore pid=5392) INFO 11-09 19:38:51 [core.py:1525] [shutdown] EngineCore: trigger received signal=SIGTERM
(EngineCore pid=5392) INFO 11-09 19:38:51 [core.py:1712] [shutdown] EngineCore: start mode=abort timeout=0s
(EngineCore pid=5392) INFO 11-09 19:38:51 [core.py:1743] [shutdown] EngineCore: request processing complete; starting resource teardown
(EngineCore pid=5392) INFO 11-09 19:38:51 [core.py:1573] [shutdown] EngineCore: exiting busy loop


exit=0  (8.5 phút)


## B3. Kết quả theo nhóm
`accuracy` = đúng hoàn toàn (G4: đã từ chối). `partial` = F1 tập/đường. `abstention_accuracy` chỉ có nghĩa ở G4,
`false_abstention_rate` ở G1-G3. `fabricated_answer_rate` = câu trả lời có tên không tồn tại trong corpus.
`ev_recall` = tài liệu chuẩn được trích dẫn; `ev_recall_visible` = tài liệu chuẩn có mặt trong prompt.

In [6]:
COLS = ['accuracy', 'partial', 'abstention_accuracy', 'false_abstention_rate', 'fabricated_answer_rate',
        'unsupported_claim_rate', 'ev_recall', 'ev_recall_visible', 'tokens_in_mean']
for rd, R in RB.items():
    acc = pd.DataFrame({NAMES.get(m, m): {g: s[g].get('accuracy') for g in ('G1', 'G2', 'G3', 'G4', 'all')}
                        for m, s in R['summaries'].items()}).T
    print(f'##### {tag(rd)}: accuracy theo nhóm'); display(acc.astype(float).round(3))
    for g in ('G1', 'G2', 'G3', 'G4'):
        df = pd.DataFrame({m: s[g] for m, s in R['summaries'].items()}).T
        print(f'--- {tag(rd)} | {g}'); display(df[[c for c in COLS if c in df.columns]].astype(float).round(3))

##### Meta-Llama-3.1-8B-Instruct: accuracy theo nhóm


,G1,G2,G3,G4,all
E0 LLM thuần,0.62,0.00,0.05,0.90,0.392
E1 RAG dense,0.63,0.03,0.02,0.86,0.385
E2 RAG hybrid,0.51,0.26,0.07,0.76,0.400
E2 BM25,0.57,0.26,0.04,0.76,0.408
E3 KG-RAG,0.26,0.33,0.20,0.93,0.430
"E3a cùng cạnh, câu rời",0.38,0.33,0.15,0.82,0.420
E3b fact chọn bằng truy hồi,0.58,0.06,0.02,0.87,0.382
E4 KG + kiểm chứng,0.26,0.33,0.20,0.93,0.430


--- Meta-Llama-3.1-8B-Instruct | G1


,accuracy,partial,abstention_accuracy,false_abstention_rate,fabricated_answer_rate,unsupported_claim_rate,ev_recall,ev_recall_visible,tokens_in_mean
e0,0.62,0.62,NaN,0.22,0.0,0.013,0.80,1.0,343.40
e1,0.63,0.63,NaN,0.21,0.0,0.000,0.80,1.0,1141.74
e2,0.51,0.51,NaN,0.37,0.0,0.003,0.64,1.0,1028.55
e2_bm25,0.57,0.57,NaN,0.31,0.0,0.038,0.67,1.0,827.73
e3,0.26,0.26,NaN,0.63,0.0,0.141,0.45,1.0,375.40
e3a_exact,0.38,0.38,NaN,0.57,0.0,0.107,0.50,1.0,373.40
e3a,0.58,0.58,NaN,0.27,0.0,0.027,0.74,1.0,900.60
e4,0.26,0.26,NaN,0.63,0.0,0.141,0.45,1.0,375.40


--- Meta-Llama-3.1-8B-Instruct | G2


,accuracy,partial,abstention_accuracy,false_abstention_rate,fabricated_answer_rate,unsupported_claim_rate,ev_recall,ev_recall_visible,tokens_in_mean
e0,0.00,0.00,NaN,1.00,0.0,1.000,0.000,0.000,276.38
e1,0.03,0.03,NaN,0.94,0.0,0.075,0.090,0.235,1732.44
e2,0.26,0.26,NaN,0.54,0.0,0.039,0.405,0.615,1702.71
e2_bm25,0.26,0.26,NaN,0.31,0.0,0.013,0.625,0.895,1530.02
e3,0.33,0.33,NaN,0.51,0.0,0.251,0.525,1.000,750.98
e3a_exact,0.33,0.33,NaN,0.53,0.0,0.310,0.510,1.000,748.48
e3a,0.06,0.06,NaN,0.90,0.0,0.571,0.215,0.690,822.88
e4,0.33,0.33,NaN,0.51,0.0,0.251,0.525,1.000,750.98


... (10 outputs left out)

--- Qwen2.5-Coder-7B-Instruct | G3


,accuracy,partial,abstention_accuracy,false_abstention_rate,fabricated_answer_rate,unsupported_claim_rate,ev_recall,ev_recall_visible,tokens_in_mean
e0,0.00,0.000,NaN,0.97,0.03,1.000,0.000,0.000,282.21
e1,0.22,0.247,NaN,0.51,0.04,0.049,0.475,0.571,1495.42
e2,0.23,0.252,NaN,0.50,0.02,0.088,0.461,0.582,1442.62
e2_bm25,0.21,0.210,NaN,0.63,0.00,0.003,0.396,0.451,1298.21
e3,0.54,0.603,NaN,0.21,0.05,0.019,0.948,0.968,676.23
e3a_exact,0.54,0.583,NaN,0.18,0.05,0.076,0.838,0.968,646.83
e3a,0.15,0.160,NaN,0.64,0.01,0.042,0.559,0.621,996.93
e4,0.52,0.583,NaN,0.28,0.05,0.019,0.948,0.968,676.23


--- Qwen2.5-Coder-7B-Instruct | G4


,accuracy,partial,abstention_accuracy,false_abstention_rate,fabricated_answer_rate,unsupported_claim_rate,ev_recall,ev_recall_visible,tokens_in_mean
e0,0.85,0.85,0.85,NaN,0.00,0.792,0.232,0.232,279.69
e1,0.59,0.59,0.59,NaN,0.00,0.096,0.853,0.894,1340.06
e2,0.47,0.47,0.47,NaN,0.01,0.115,0.814,0.942,1322.11
e2_bm25,0.50,0.50,0.50,NaN,0.02,0.120,0.795,0.959,1148.51
e3,0.56,0.56,0.56,NaN,0.00,0.052,0.990,1.000,671.42
e3a_exact,0.49,0.49,0.49,NaN,0.00,0.175,0.860,1.000,649.53
e3a,0.56,0.56,0.56,NaN,0.00,0.081,0.879,0.894,892.79
e4,0.65,0.65,0.65,NaN,0.00,0.052,0.990,1.000,671.42


## B4. So sánh cặp (cùng câu hỏi), bootstrap theo cụm thực thể + McNemar
Một cặp chỉ được nói là khác nhau khi khoảng tin cậy 95% không chứa 0. Kết quả âm hoặc không có ý nghĩa vẫn là kết quả.

In [7]:
for rd, R in RB.items():
    rows = [dict(pair=k.split('/')[0], group=c['group'], question=c['question'], n=c['n'], delta=round(c['delta'], 3),
                 ci=f"[{c['ci_low']:+.3f}, {c['ci_high']:+.3f}]", mcnemar_p=round(c['mcnemar_p'], 4), sig=c['significant_0.05'])
            for k, c in R['comparisons'].items()]
    print('#####', tag(rd)); display(pd.DataFrame(rows))

##### Meta-Llama-3.1-8B-Instruct


,pair,group,question,n,delta,ci,mcnemar_p,sig
0,e1_vs_e0,all,documents vs LLM only,400,-0.008,"[-0.047, +0.032]",0.7838,False
1,e1_vs_e0,G1,documents vs LLM only,100,0.010,"[-0.087, +0.112]",1.0000,False
2,e1_vs_e0,G2,documents vs LLM only,100,0.030,"[+0.000, +0.072]",0.2500,False
3,e1_vs_e0,G3,documents vs LLM only,100,-0.030,"[-0.080, +0.020]",0.4531,False
4,e1_vs_e0,G4,documents vs LLM only,100,-0.040,"[-0.133, +0.052]",0.4545,False
5,e2_vs_e1,all,hybrid vs dense,400,0.015,"[-0.042, +0.065]",0.5764,False
6,e2_vs_e1,G1,hybrid vs dense,100,-0.120,"[-0.210, -0.030]",0.0227,True
7,e2_vs_e1,G2,hybrid vs dense,100,0.230,"[+0.129, +0.423]",0.0000,True
8,e2_vs_e1,G3,hybrid vs dense,100,0.050,"[-0.008, +0.128]",0.1797,False
9,e2_vs_e1,G4,hybrid vs dense,100,-0.100,"[-0.198, +0.000]",0.0414,False


##### Qwen2.5-Coder-7B-Instruct


,pair,group,question,n,delta,ci,mcnemar_p,sig
0,e1_vs_e0,all,documents vs LLM only,400,0.092,"[-0.002, +0.180]",0.0003,False
1,e1_vs_e0,G1,documents vs LLM only,100,0.050,"[-0.032, +0.135]",0.3018,False
2,e1_vs_e0,G2,documents vs LLM only,100,0.360,"[+0.210, +0.465]",0.0000,True
3,e1_vs_e0,G3,documents vs LLM only,100,0.220,"[+0.067, +0.364]",0.0000,True
4,e1_vs_e0,G4,documents vs LLM only,100,-0.260,"[-0.375, -0.145]",0.0000,True
5,e2_vs_e1,all,hybrid vs dense,400,0.000,"[-0.047, +0.046]",1.0000,False
6,e2_vs_e1,G1,hybrid vs dense,100,-0.050,"[-0.118, +0.011]",0.2266,False
7,e2_vs_e1,G2,hybrid vs dense,100,0.160,"[+0.043, +0.373]",0.0113,True
8,e2_vs_e1,G3,hybrid vs dense,100,0.010,"[-0.083, +0.109]",1.0000,False
9,e2_vs_e1,G4,hybrid vs dense,100,-0.120,"[-0.192, -0.062]",0.0042,True


## B5. Lỗi theo bẫy và theo loại câu hỏi
Bẫy nào làm method nào sai: `superseded_policy` (đọc nhầm policy cũ), `dev_only` / `extra_not_enabled` (đếm cả đường không triển khai),
`version_out_of_range`, `decommissioned`, `unknown_entity` / `unpinned` / `conflicting_policies` (lẽ ra phải từ chối).

In [8]:
for rd, R in RB.items():
    T = pd.DataFrame({m: {t: v['accuracy'] for t, v in s['by_trap'].items()} for m, s in R['summaries'].items()})
    T.insert(0, 'n', [R['summaries'][SC_METHODS[0]]['by_trap'][t]['n'] for t in T.index])
    print('#####', tag(rd), '| accuracy theo bẫy'); display(T.round(2))
    Q = pd.DataFrame({m: {t: v['accuracy'] for t, v in s['by_type'].items()} for m, s in R['summaries'].items()})
    print('#####', tag(rd), '| accuracy theo loại câu hỏi'); display(Q.round(2))

##### Meta-Llama-3.1-8B-Instruct | accuracy theo bẫy


,n,e0,e1,e2,e2_bm25,e3,e3a_exact,e3a,e4
adr_exception,3,0.00,0.00,0.00,0.00,0.33,0.67,0.00,0.33
adr_expired,2,0.00,0.00,0.50,1.00,1.00,0.50,0.00,1.00
aggregation,21,0.24,0.05,0.00,0.05,0.10,0.00,0.00,0.10
co_occurrence,17,0.29,0.18,0.18,0.29,0.06,0.12,0.18,0.06
config_plugins,12,1.00,0.75,0.75,0.83,1.00,1.00,1.00,1.00
conflicting_policies,17,1.00,0.76,0.76,0.12,1.00,0.41,0.94,1.00
cross_function,39,0.74,0.77,0.67,0.64,0.28,0.33,0.59,0.28
decommissioned,3,0.00,0.00,0.00,0.00,0.33,0.67,0.33,0.33
dev_exempt,3,0.00,0.00,0.33,0.33,0.33,0.33,0.33,0.33
dynamic_dispatch,9,0.44,0.78,0.22,0.44,0.78,0.67,0.78,0.78


##### Meta-Llama-3.1-8B-Instruct | accuracy theo loại câu hỏi


,e0,e1,e2,e2_bm25,e3,e3a_exact,e3a,e4
G1/code_env,1.00,0.80,0.80,1.00,0.20,0.80,1.00,0.20
G1/code_flow,0.56,0.64,0.48,0.49,0.25,0.32,0.57,0.25
G1/code_host,0.78,0.56,0.56,0.78,0.33,0.50,0.50,0.33
G2/ban_allowed,0.00,0.02,0.10,0.23,0.23,0.29,0.06,0.23
G2/exception_valid,0.00,0.25,0.25,1.00,1.00,0.75,0.00,1.00
G2/patch_window,0.00,0.00,0.65,0.35,0.48,0.43,0.04,0.48
G2/token_lifetime,0.00,0.09,0.36,0.00,0.36,0.18,0.09,0.36
G3/affected_filtered,0.71,0.14,0.00,0.14,0.00,0.00,0.00,0.00
G3/affected_set,0.00,0.00,0.00,0.00,0.29,0.00,0.00,0.29
G3/exposure_path,0.00,0.00,0.00,0.00,0.46,0.46,0.00,0.46


##### Qwen2.5-Coder-7B-Instruct | accuracy theo bẫy


,n,e0,e1,e2,e2_bm25,e3,e3a_exact,e3a,e4
adr_exception,3,0.00,0.33,0.33,0.33,1.00,1.00,0.00,1.00
adr_expired,2,0.00,1.00,1.00,0.00,0.00,0.00,1.00,0.00
aggregation,21,0.00,0.00,0.00,0.00,0.19,0.10,0.00,0.10
co_occurrence,17,0.29,0.29,0.29,0.29,0.35,0.35,0.29,0.35
config_plugins,12,1.00,0.33,0.17,0.08,1.00,0.75,0.33,1.00
conflicting_policies,17,1.00,0.82,0.35,0.12,0.00,0.00,0.59,0.00
cross_function,39,0.79,0.90,0.82,0.67,0.79,0.79,0.90,0.79
decommissioned,3,0.00,0.33,0.33,0.67,1.00,0.33,0.00,1.00
dev_exempt,3,0.00,0.67,0.00,0.33,1.00,1.00,0.67,1.00
dynamic_dispatch,9,0.11,0.00,0.00,0.00,0.67,0.67,0.11,0.67


##### Qwen2.5-Coder-7B-Instruct | accuracy theo loại câu hỏi


,e0,e1,e2,e2_bm25,e3,e3a_exact,e3a,e4
G1/code_env,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
G1/code_flow,0.68,0.74,0.68,0.56,0.65,0.64,0.73,0.65
G1/code_host,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
G2/ban_allowed,0.00,0.47,0.44,0.45,0.40,0.48,0.45,0.35
G2/exception_valid,0.00,0.75,1.00,0.75,0.75,0.75,1.00,0.75
G2/patch_window,0.00,0.00,0.61,0.26,0.87,0.70,0.09,0.87
G2/token_lifetime,0.00,0.36,0.64,0.73,1.00,0.82,0.18,1.00
G3/affected_filtered,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
G3/affected_set,0.00,0.00,0.00,0.00,0.29,0.14,0.00,0.29
G3/exposure_path,0.00,0.00,0.00,0.00,0.62,0.92,0.00,0.62


## B6. Ví dụ (một câu mỗi nhóm): câu trả lời của từng method

In [9]:
from pypi_rag.schema import read_jsonl
rd = READERS[0]
QS = {q['qid']: q for q in read_jsonl(f'{SC}/test/questions.jsonl')}
PR = {m: {r['qid']: r for r in read_jsonl(f'{SC}/runs/test_{tag(rd)}/predictions_{m}.jsonl')} for m in SC_METHODS}
for g, trap in (('G1', 'co_occurrence'), ('G2', 'superseded_policy'), ('G3', 'extra_not_enabled'), ('G4', 'unpinned')):
    q = next((q for q in QS.values() if q['group'] == g and q['trap'] == trap), None) or next(q for q in QS.values() if q['group'] == g)
    print(f"\n===== {g} | {q['qtype']} | bẫy {q['trap']}\n{q['question']}\nđáp án chuẩn: {q['gold']}")
    for m in SC_METHODS:
        r = PR[m][q['qid']]
        ans = 'insufficient_evidence' if r['insufficient'] else r['answer']
        print(f"  {NAMES[m]:30s} -> {str(ans)[:80]}   (claims {r['check']['n_supported']}/{r['check']['n_claims']} có chứng cứ)")


===== G1 | code_flow | bẫy co_occurrence
Does data read from the local machine (environment variables, files, host or user names) reach a network destination in this code?
đáp án chuẩn: yes
  E0 LLM thuần                   -> no   (claims 5/5 có chứng cứ)
  E1 RAG dense                   -> no   (claims 5/5 có chứng cứ)
  E2 RAG hybrid                  -> no   (claims 5/5 có chứng cứ)
  E2 BM25                        -> insufficient_evidence   (claims 0/0 có chứng cứ)
  E3 KG-RAG                      -> insufficient_evidence   (claims 0/0 có chứng cứ)
  E3a cùng cạnh, câu rời         -> insufficient_evidence   (claims 0/0 có chứng cứ)
  E3b fact chọn bằng truy hồi    -> insufficient_evidence   (claims 0/0 có chứng cứ)
  E4 KG + kiểm chứng             -> insufficient_evidence   (claims 0/0 có chứng cứ)

===== G2 | ban_allowed | bẫy superseded_policy
Under ACME's current security policy, may svc-checkout deploy sniffio==1.13.0 as a runtime dependency?
đáp án chuẩn: yes
  E0 LLM thuần   

## B7. Kết luận tự động (chỉ nêu điều có kiểm định)

In [10]:
for rd, R in RB.items():
    print('#####', tag(rd))
    for key, txt in [('e1_vs_e0', 'Tài liệu (RAG) so với LLM thuần'), ('e3_vs_e2', 'KG so với RAG hybrid'),
                     ('e3_vs_e3a_exact', 'Cấu trúc đường so với cùng cạnh dạng câu rời'), ('e4_vs_e3', 'Kiểm chứng so với không')]:
        for g in ('G1', 'G2', 'G3', 'G4'):
            c = R['comparisons'].get(f'{key}/{g}')
            if not c: continue
            verdict = ('tốt hơn' if c['delta'] > 0 else 'kém hơn') if c['significant_0.05'] else 'không khác biệt có ý nghĩa'
            print(f"  {txt} | {g}: {verdict} (delta {c['delta']:+.3f}, CI [{c['ci_low']:+.3f}, {c['ci_high']:+.3f}])")

##### Meta-Llama-3.1-8B-Instruct
  Tài liệu (RAG) so với LLM thuần | G1: không khác biệt có ý nghĩa (delta +0.010, CI [-0.087, +0.112])
  Tài liệu (RAG) so với LLM thuần | G2: không khác biệt có ý nghĩa (delta +0.030, CI [+0.000, +0.072])
  Tài liệu (RAG) so với LLM thuần | G3: không khác biệt có ý nghĩa (delta -0.030, CI [-0.080, +0.020])
  Tài liệu (RAG) so với LLM thuần | G4: không khác biệt có ý nghĩa (delta -0.040, CI [-0.133, +0.052])
  KG so với RAG hybrid | G1: kém hơn (delta -0.250, CI [-0.379, -0.126])
  KG so với RAG hybrid | G2: không khác biệt có ý nghĩa (delta +0.070, CI [-0.129, +0.180])
  KG so với RAG hybrid | G3: tốt hơn (delta +0.130, CI [+0.020, +0.247])
  KG so với RAG hybrid | G4: tốt hơn (delta +0.170, CI [+0.054, +0.283])
  Cấu trúc đường so với cùng cạnh dạng câu rời | G1: kém hơn (delta -0.120, CI [-0.219, -0.021])
  Cấu trúc đường so với cùng cạnh dạng câu rời | G2: không khác biệt có ý nghĩa (delta +0.000, CI [-0.075, +0.119])
  Cấu trúc đường so với cùng cạ